## RAG 시스템 구축
 - RAG 시스템을 구축하는 것은 쉬우나 실제로 프로덕트 개발을 위해 RAG 품질을 높이려면 고급적인 기법이 필요하다.
 - Retriever: 사용자의 질문의 의도를 정확하게 파악하여 고품질의 답변을 하는것이 필수적
 - 1. 대충 질문해도 품질 좋은 답변을 원함
 - 2. 앞뒤 문맥을 잘 파악한 답변을 원함
 - 3. 시멘틱 검색말고 쿼리가 필요
 - 4. 오래된 자료는 덜 참고


## Multi-Query Retriever
  - 질문이 추상적이거나 좋지 못해도, 사용자 질문을 여러개의 유사 질문으로 재생성
  - ex) A은행의 대출은 어때? ----> 1. A은행의 대출 금리는 어때? 2. A은행의 대출 조건은 어때? 3. A은행의 대출 후기는 어때?

In [1]:
!pip install -q langchain pypdf sentence-transformers chromadb openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.0/329.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 5.8 MB/s eta

In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface chromadb

# Multi-Query Retriever

In [7]:
# 1. 텍스트 분할기 (기본 langchain 패키지에 포함)
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 2. 문서 로더 (community 패키지)
from langchain_community.document_loaders import WebBaseLoader

# 3. 벡터 저장소 (Chroma)
from langchain_community.vectorstores import Chroma

# 4. 임베딩 (HuggingFace)
# 최신 버전에서는 langchain_community 또는 langchain_huggingface를 사용합니다.
from langchain_community.embeddings import HuggingFaceEmbeddings

# load website post
loader = WebBaseLoader('https://n.news.naver.com/article/088/0000991940?sid=100')
data = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0)
splits = text_splitter.split_documents(data)

# VectorDB
model_name = 'jhgan/ko-sbert-nli'
encode_kwargs = {'normalize_embeddings': True}
ko_embedding = HuggingFaceEmbeddings(
    model_name=model_name,
    encode_kwargs=encode_kwargs
)

vectordb = Chroma.from_documents(documents=splits, embedding=ko_embedding)

In [ ]:
!pip install langchain==0.1.16

In [8]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.chat_models import ChatOpenAI

question = "한동훈 제명이야?"
llm = ChatOpenAI(temperature=0, openai_api_key='')
retriever_from_llm = MultiQueryRetriever.from_llm(
    retriever=vectordb.as_retriever(),
    llm=llm
)

In [9]:
import logging

logging.basicConfig()
logging.getLogger('langchain.retrievers.multi_query').setLevel(logging.INFO)

In [10]:
unique_docs = retriever_from_llm.get_relevant_documents(query=question)
len(unique_docs)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. 한동훈이 제명되었나요?', '2. 한동훈이 제명당했나요?', '3. 한동훈의 제명 여부를 알고 싶어요.']


5

In [11]:
unique_docs

[Document(page_content='단체에 비견될 정도로 중대한 사안"이라고 지적했다. 이와 관련 한 전 대표는 "윤리위원이 무슨 국정원 블랙 요원인가"라며 "윤리위원장이 어떤 사람인지를 왜 우리가 몰라야 하나"라고 맞받았다.한 전 대표는 장동혁 대표가 윤리위의 제명 결정을 \'독자적인 판단\'이라고 한 것과 관련 "솔직해지자. 장 대표가 계엄을 막아낸 저를 찍어내기 위한 일을 하는 것"이라고 지적했다.당원 게시판 사건은 지난해 11월 국민의힘 당원 게시판에 올라온 윤석열 전 대통령 부부 비방 글 등이 한 전 대표 가족 명의로 작성됐다는 의혹이다. 이후 국민의힘은', metadata={'source': 'https://n.news.naver.com/article/088/0000991940?sid=100', 'language': 'ko', 'title': '한동훈 "윤리위 제명은 또 다른 계엄…장동혁이 나를 찍어내"'}),
 Document(page_content='국민의힘 한동훈 전 대표가 14일 국회 소통관에서 당 윤리위원회가 본인을 제명 결정한 것과 관련해 입장을 밝히기 위해 입장해 있다. 연합뉴스한동훈 전 국민의힘 대표는 국민의힘 윤리위원회가 \'당원게시판 사건\'으로 자신을 제명하자, "계엄을 극복하고 통합해야 할 때 헌법과 민주주의를 파괴하는 또 다른 계엄이 선포된 것"이라고 14일 말했다.한 전 대표는 이날 오후 1시 30분 국회에서 긴급 기자회견을 열어 "윤리위는 계엄을 막고 당을 지킨 저를 허위 조작으로 제명했다"며 이 같이 주장했다. 한 대표는 "국민, 당원과 함께 이번 계엄도', metadata={'source': 'https://n.news.naver.com/article/088/0000991940?sid=100', 'title': '한동훈 "윤리위 제명은 또 다른 계엄…장동혁이 나를 찍어내"', 'language': 'ko'}),
 Document(page_content='김경 \'자수서\'에 "1억원 줄 때 강선우 함께 있었다"\